## Data Loading and Chunking

In [1]:
from grequests import download

download("Chapter03","scenario.csv")

Downloaded scenario.csv to Download\scenario.csv


In [1]:
import time 
start_time = time.time()
file_path = 'scenario.csv'

# Read the file Skip the header 
chunks = [] 
with open('scenario.csv' , 'r') as file:
    next(file)     # Skip the header line
    chunks = [line.strip() for line in file]

print("Total Number of chunks :- " , len(chunks))

response_time = time.time() - start_time
print(f"Response Time: {response_time:.2f} seconds")

for i, chunk in enumerate(chunks[:3], start=1):
    print(chunk)

Total Number of chunks :-  3
Response Time: 0.01 seconds
100,Semantic analysis.This is not an analysis but a semantic search. Provide more information on the topic.
200,Sentiment analysis  Read the content return a sentiment analysis nalysis on this text and provide a score with the label named : Sentiment analysis score followed by a numerical value between 0 and 1  with no + or - sign and  add an explanation to justify the score.
300,Semantic analysis.This is not an analysis but a semantic search. Provide more information on the topic.


## Embedding the Dataset

In [2]:
import openai
import time 
import os 
from dotenv import load_dotenv

load_dotenv()

# Setup Embeddings model
embedding_model="text-embedding-3-small"
client = openai.OpenAI(api_key=os.getenv('open_ai_api'))

# Embeddings function
def get_embeddings(texts , model = embedding_model):
    texts = [text.replace("\n" , " ") for text in texts] # Clean the text
    response = client.embeddings.create(input = texts , model=model)
    embeddings = [res.embedding for res in response.data]
    return embeddings

In [3]:
def embed_chunks(chunks , embedding_model = 'text-embedding-3-small',batch_size = 1000 , pause_time = 3):
    start_time = time.time()
    embeddings = []
    counter = 1

    # Processing chunks in batches
    for i in range(0 , len(chunks) , batch_size):
        chunk_batch = chunks[i : i+batch_size]

        # Get the embeddings for the current batch
        current_embeddings = get_embeddings(chunk_batch , model = embedding_model)

        # Adding to the final list
        embeddings.extend(current_embeddings)

        # Print the batch process
        print(f"Batch {counter} embedded.")
        counter += 1
        time.sleep(pause_time)
    
    # Print total Response time 
    response_time = time.time() - start_time
    print("Total Response time :- " + f"{response_time:.2f}" + " seconds")

    return embeddings

embeddings = embed_chunks(chunks)

Batch 1 embedded.
Total Response time :- 6.78 seconds


In [4]:
print("First Embeddings :\n" , embeddings[0])

First Embeddings :
 [0.0178070068359375, 0.041595458984375, -0.02410888671875, -0.0214385986328125, 0.0013790130615234375, -0.00844573974609375, -0.00948333740234375, 0.0143585205078125, 0.06427001953125, -0.006954193115234375, -0.0005269050598144531, -0.06591796875, -0.044464111328125, -0.0294647216796875, 0.0021877288818359375, -0.01319122314453125, 0.0140533447265625, 0.0294342041015625, 0.064697265625, 0.04180908203125, 0.07611083984375, 0.018646240234375, -0.035186767578125, 0.060272216796875, 0.053497314453125, -0.0504150390625, 0.00643157958984375, 0.036956787109375, 0.026214599609375, -0.0301055908203125, 0.0128173828125, -0.0305023193359375, 0.01995849609375, -0.01198577880859375, -0.029693603515625, 0.0128021240234375, -0.061065673828125, -0.0240478515625, 0.00457000732421875, -0.0166473388671875, -0.046295166015625, -0.029632568359375, 0.00038051605224609375, -0.01491546630859375, -0.0016965866088867188, -0.01270294189453125, -0.02203369140625, -0.020263671875, 0.00457382202

In [5]:
num_chunks = len(chunks)
print(f"Number of chunks: {num_chunks}")
print(f"Number of embeddings: {len(embeddings)}")

Number of chunks: 3
Number of embeddings: 3


## Pinecone Index

In [9]:
import os 
from pinecone import Pinecone, ServerlessSpec

pinecone_api = os.getenv("pinecone_api")
pc = Pinecone(api_key=pinecone_api)

In [10]:
index_name = "genai-v1"
namespace = 'genaisys'
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'
spec = ServerlessSpec(cloud=cloud, region=region)

In [12]:
import time 
import pinecone 

if index_name not in pc.list_indexes().names():
    # If the index dont exists
    pc.create_index(
        index_name,
        dimension=1536,
        metric='cosine',
        spec=spec
    )

    # Wait for index to initialise
    time.sleep(1)

index = pc.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)

In [13]:
index_info = pc.describe_index(index_name)
print(f"Cloud provider: {index_info['spec']['serverless']['cloud']}")
print(f"Cloud provider: {index_info['spec']['serverless']['region']}")

Cloud provider: aws
Cloud provider: us-east-1


### Upserting

In [14]:
import sys 
start_time = time.time()

# Function to calculate the size of the batch 
def get_batch_size(data , limit = 4000000): # Limit is 4 MB
    total_size = 0
    batch_size = 0
    for item in data:
        item_size = sum([sys.getsizeof(v) for v in item.values()])
        if total_size + item_size > limit:
            break
        total_size += item_size
        batch_size += 1 
    return batch_size

# Upsert Function with namespace
def upsert_to_pinecone(batch , batch_size , namespace = "genaisys"):
    try:
        index.upsert(vectors=batch , namespace=namespace)
        print(f"Upserted {batch_size} vectors to namespace '{namespace}'.")
    except Exception as e:
        print("Error in upserting the data to pinecone")

# Function to upsert data in batches
def batch_upsert(data):
    total = len(data)
    i = 0
    while i < total:
        batch_size = get_batch_size(data[i:])
        batch = data[i : i + batch_size]
        if batch:
            upsert_to_pinecone(batch , batch_size , "genaisys")
            i += batch_size
            print(f"Upserted {i}/{total} items...")
        else:
            break

    print("Upsert Complete.")

# Generate Id for each data
ids = [str(i) for i in range(1,len(chunks) + 1)]

# Prepare Data for upsert 
data_for_upsert = [
    {'id' : str(id) , 'values' : emb , 'metadata' : {'text' : chunk}} 
    for id, (chunk, emb) in zip(ids, zip(chunks, embeddings))
]

# Upsert data in batches
batch_upsert(data_for_upsert)

response_time = time.time() - start_time  # Measure response time
print(f"Upsertion response time: {response_time:.2f} seconds")

Upserted 3 vectors to namespace 'genaisys'.
Upserted 3/3 items...
Upsert Complete.
Upsertion response time: 1.93 seconds


In [16]:
print("Index stats")
print(index.describe_index_stats())

Index stats
DescribeIndexStatsResponse(dimension=1536, total_vector_count=3, metric='cosine', namespaces=1)
